# py4vasp quick start

Post-processing of a VASP run through `vaspout.h5` (the HDF5 output).

Requirements
* VASP compiled with `-DVASP_HDF5` (VASP >= 6.2; the repository targets 6.4-6.6)
* `pip install py4vasp` (0.11.x at the time of writing); `pip install kaleido` to save static images
* Start Jupyter in the directory of the calculation, or change `path` below.

Every quantity offers `read()` (Python dict), `plot()` (interactive figure), `selections()`
(what can be selected) and, for x-y data, `to_csv(filename=...)` / `to_image(filename=...)`.

In [ ]:
import py4vasp
print("py4vasp", py4vasp.__version__)

path = "."                      # directory that contains vaspout.h5
calc = py4vasp.Calculation.from_path(path)

## Structure
Final structure of the run; `to_POSCAR()` gives a string that can be written to a new POSCAR,
`to_ase()` converts to an ASE `Atoms` object.

In [ ]:
print(calc.structure)
atoms = calc.structure.to_ase()
print(atoms.get_chemical_formula(), atoms.cell.lengths())
with open("POSCAR_from_h5", "w") as fh:
    fh.write(calc.structure.to_POSCAR())

## Energies
Without a slice the last ionic step is used; `calc.energy[:]` selects all steps
(useful to follow a relaxation or an MD run).

In [ ]:
print(calc.energy.read())
calc.energy[:].plot()

## Band structure
For a line-mode run (or a run with KPOINTS_OPT, VASP >= 6.3) the bands are plotted along the path.
With LORBIT = 11 a selection such as `"Si(p)"`, `"1(d)"` or `"s, p"` draws fat bands.

In [ ]:
print(calc.band.selections())
calc.band.plot()
# calc.band.plot("kpoints_opt")      # bands of the KPOINTS_OPT path
# calc.band.plot("Si(s), Si(p)")     # projected (fat) bands, needs LORBIT

## Density of states and band gap

In [ ]:
calc.dos.plot()
# calc.dos.plot("s, p, d")           # orbital projections, needs LORBIT
# calc.dos.to_csv(filename="dos.csv")
calc.bandgap.print()

## Simulated STM (VASP >= 6.5, LPARD = .TRUE. and LPARDH5 = .TRUE.)
Run this cell in the directory of the partial-charge calculation.

In [ ]:
stm = calc.partial_density.to_stm("constant_height", tip_height=3.0, supercell=5)
stm.show()

## Saving figures
`calc.dos.to_image(filename="dos.png")` writes a PNG (requires kaleido);
`calc.band.to_csv(filename="bands.csv")` exports the data for Origin/Excel/matplotlib.

In [ ]:
calc.dos.to_image(filename="dos.png")
calc.band.to_csv(filename="bands.csv")